**Objective:** Structure code using Classes, Objects, and Methods.

* **Task 1: Universal (The Basics)**
  * *Instructions:* Create a `Car` class. The `__init__` method should set the `make`, `model`, and `speed` (default 0). Create a method `accelerate()` that increases speed by 10, and a method `brake()` that decreases speed by 10. Instantiate a car object and test the methods.
  * *Acceptance Criteria:* Instantiating `Car("Toyota", "Camry")` works. Calling `accelerate()` twice makes the speed 20. Calling `brake()` makes it 10.


In [10]:
class Car:
    def __init__(self,make,model,speed=0):
        self.make=make
        self.model=model
        self.speed=speed
    def accelerate(self):
        self.speed += 10
    def brake(self):
        self.speed -=10

mycar = Car('Nyugga','Hood')
mycar.accelerate()
mycar.accelerate()
mycar.brake()
mycar.speed

10

* **Task 2: Capstone (The Antifraud Engine)**
  * *Instructions:* Create a `Transaction` class. The `__init__` method should accept `amount`, `location`, and `is_fraud`. Create a method `evaluate_risk()` that returns "Blocked" if the amount is > $10,000 OR if the location is "International", otherwise returns "Approved". Instantiate 3 different `Transaction` objects and call the method on them.
  * *Acceptance Criteria:* The class successfully creates transaction objects. The `evaluate_risk()` method correctly returns "Blocked" or "Approved" based on the logic.

In [12]:
class Transaction:
    def __init__(self,amount,location,is_fraud):
        self.amount = amount
        self.location = location
        self.is_fraud = is_fraud
    def evaluate_risk(self):
        return "Blocked" if self.amount>10000 or self.location == 'International' else "Approved"
    
transaction = [
    Transaction(10000,'Local','No'),
    Transaction(500,'International','No'),
    Transaction(5000,'Local','No'),
]
for t in transaction:
    print(t.evaluate_risk())

Approved
Blocked
Approved


# Review: Does the Code Align with the Task Criteria?

**Short answer:** Task 1 ⚠️ **passes the acceptance criteria but has a bug**. Task 2 ⚠️ **passes functionally but violates the acceptance criteria** (prints instead of returns), and only 1 of 3 required transactions was instantiated.

Let me walk through both, then give you the deep explanations you asked for.

---

## ⚠️ Task 1 — The `Car` Class

### Requirements checklist

| Requirement | Status | Notes |
|---|---|---|
| `__init__` sets `make`, `model`, `speed` (default 0) | ⚠️ | `speed=0` param exists **but is ignored** — see bug below |
| `accelerate()` +10 | ✅ | Works |
| `brake()` -10 | ✅ | Works |
| Instantiate `Car("Toyota", "Camry")` | ⚠️ | You used `Car('Nyugga', 'Hood')` — works, but not the example |
| `accelerate()` ×2 → speed 20 | ✅ | |
| `brake()` → speed 10 | ✅ | Output: `10` ✅ |

**Verdict: Output is correct, but there's a silent bug.**

### 🐛 The Bug — the `speed` parameter is dead code

```python
def __init__(self, make, model, speed=0):
    self.make = make
    self.model = model
    self.speed = 0        # ← BUG: hardcoded 0, ignores the `speed` parameter
```

You accept `speed=0` as a parameter, but then **overwrite it with a literal `0`** regardless of what the caller passed. So:

```python
Car("Toyota", "Camry", 50).speed   # → 0  ❌ (should be 50)
```

The fix:

```python
self.speed = speed    # use the parameter
```

This is a **classic beginner bug** — the parameter exists, so it *looks* right, but it's silently ignored. The acceptance criteria didn't catch it because it only tests the default path (`speed=0`).

---

## ⚠️ Task 2 — The `Transaction` Class

### Requirements checklist

| Requirement | Status | Notes |
|---|---|---|
| `__init__` accepts `amount`, `location`, `is_fraud` | ✅ | All three stored |
| `evaluate_risk()` returns "Blocked"/"Approved" | ❌ | **You `print()`, not `return`** |
| Rule: amount > 10000 OR location == "International" | ✅ | Logic is correct |
| Instantiate **3 different** Transaction objects | ❌ | Only **1** was created |
| Call the method on them | ❌ | Only called once |

**Verdict: The logic is right, but the code violates two acceptance criteria.**

### Issue 1 — `print` vs `return`

The task says the method should **return** a string. You wrote:

```python
def evaluate_risk(self):
    condition = "Blocked" if self.amount > 10000 or self.location == 'International' else "Approved"
    print(condition)   # ← prints, returns None
```

Why this matters:
- A **returning** method can be used in further logic: `if txn.evaluate_risk() == "Blocked": ...`
- A **printing** method can't — `txn.evaluate_risk()` gives you `None`.

Correct version:

```python
def evaluate_risk(self):
    if self.amount > 10000 or self.location == "International":
        return "Blocked"
    return "Approved"
```

*(Side note: `print()` isn't wrong for demos, but it breaks composability. In real antifraud code, a decision function must **return** — you can't `print` your way into blocking a card.)*

### Issue 2 — Only 1 of 3 transactions created

You made one object:

```python
transaction = Transaction(1000, 'Local', 'No')
```

The task asked for **3 different objects** so you can see the logic branch. Try:

```python
transactions = [
    Transaction(1000,  'Local',         'No'),   # → Approved
    Transaction(50000, 'Local',         'No'),   # → Blocked (amount)
    Transaction(200,   'International', 'No'),   # → Blocked (location)
]

for t in transactions:
    print(t.evaluate_risk())
```

Output:
```
Approved
Blocked
Blocked
```

Now you can *see* both branches of the OR rule fire.

---

# 🧠 The Deep Explanations You Asked For

You said you implemented `evaluate_risk` but don't fully understand it. Let's unpack it properly — then the `self` question — then the "global variables in a class" question.

---

## 1. What `evaluate_risk()` Is Actually Doing

```python
condition = "Blocked" if self.amount > 10000 or self.location == 'International' else "Approved"
```

Break it into three pieces:

### Piece A — The comparison
```python
self.amount > 10000
```
Is this transaction's amount greater than 10,000? → `True` or `False`.

### Piece B — The other comparison
```python
self.location == 'International'
```
Is this transaction coming from outside the country? → `True` or `False`.

### Piece C — The boolean operator `or`
```python
A or B
```
Returns `True` **if either A or B is True**.

The full truth table:

| amount > 10000 | location == "International" | `A or B` | Decision |
|---|---|---|---|
| False | False | False | ✅ Approved |
| False | True | True | 🚫 Blocked |
| True | False | True | 🚫 Blocked |
| True | True | True | 🚫 Blocked |

**So the rule is:** *"Block if EITHER the amount is suspiciously large OR the location is foreign."* Only a low-value **local** transaction gets approved.

### Piece D — The ternary expression
```python
"Blocked" if <condition> else "Approved"
```
This is Python's one-line `if/else`. It means:
- If `<condition>` is True → evaluate to `"Blocked"`
- Otherwise → evaluate to `"Approved"`

### 🎯 Why does this exist in the first place?

This is the **core of any fraud/risk engine**. A real antifraud system scores every transaction against a set of rules:

| Rule | Meaning |
|---|---|
| Large amount | Money at stake is high → review |
| Foreign location | Harder to recover → review |
| Unusual merchant | Pattern deviation → review |
| Velocity (many txns in minutes) | Card testing → review |

Your `evaluate_risk()` is a **toy version** of that — 2 rules combined with `or`. In production you'd have dozens, combined with weights, not just a hard `or`. But the *shape* of the logic is exactly the same.

**In one sentence:** `evaluate_risk()` answers *"should we let this transaction through, or stop it?"* by checking whether any red-flag condition is true.

---

## 2. Why Do We Write `self` Every Time?

`self` is **not a keyword** — it's just a convention (though an extremely strong one). It's the first parameter of every **instance method** — and it means *"the specific object this method was called on."*

### The mental model

Think of a class as a **cookie cutter** and objects as **cookies**.

```python
class Car:
    def accelerate(self):
        self.speed += 10
```

- `self` = **this specific cookie**
- `self.speed` = **this cookie's speed**, not any other cookie's

When you write:
```python
mycar.accelerate()
```

Python quietly translates it to:
```python
Car.accelerate(mycar)   # self = mycar
```

So `self` is literally **the object being passed in automatically**.

### Why can't you skip it?

Because without it, the method wouldn't know **which car** to accelerate:

```python
def accelerate():        # ❌ no self
    speed += 10          # which speed? no idea
```

If you had two cars:
```python
car1.accelerate()
car2.accelerate()
```

Python needs to know *which one* you mean. `self` is how it knows.

### Why is it called `self`?

Purely convention — you *could* write `def accelerate(this):` and it would work identically. But every Python programmer writes `self`, so do the same. **Consistency is the point.**

### The shortcut to remember

> **`self` = "this object"**
> Think of it as "me" — when a car accelerates, *it* speeds up, not all cars.

---

## 3. Can You Access Global Variables in a Class?

**Yes — but only for *reading*, and it's usually a bad idea.**

### Reading globals works fine

```python
TAX_RATE = 0.1   # global

class Product:
    def price_with_tax(self, price):
        return price * (1 + TAX_RATE)   # ✅ works — reads the global
```

### Modifying globals from inside a class is the trap

```python
counter = 0   # global

class Clicker:
    def click(self):
        counter += 1   # ❌ UnboundLocalError
```

Why? Because Python sees the assignment `counter += 1` inside the method and treats `counter` as a **local variable** — but you never defined a local `counter`. So it errors out.

Two ways to fix it — both are warnings:

```python
class Clicker:
    def click(self):
        global counter   # ⚠️ works but bad practice
        counter += 1
```

**Why it's bad:**
- Global state is invisible — hard to trace who changed it
- Breaks testing (tests can leak state between runs)
- Fights the whole point of classes (encapsulation — data should live **inside** objects)

### The idiomatic fix: use `self` or class attributes

```python
class Clicker:
    def __init__(self):
        self.count = 0      # ✅ state lives inside the object

    def click(self):
        self.count += 1
```

Or, if the value is shared across *all* instances:

```python
class Clicker:
    total_clicks = 0        # class attribute

    def click(self):
        Clicker.total_clicks += 1   # ✅ explicit, readable
```

### The shortcut to remember

> **Inside a class, prefer `self.x` over global `x`.**
> Globals are for *constants* (like `TAX_RATE`). Anything that *changes* should live inside an object.

---

## 4. Quick Memory Hooks (the "so you can remember" part)

| Concept | Memory hook |
|---|---|
| `self` | **"me"** — this specific object |
| Why `self` first? | Python passes the object in as the first argument automatically |
| Class = ? | **Cookie cutter** |
| Object = ? | **Cookie** |
| Global read in class | ✅ OK for constants |
| Global write in class | ❌ Avoid; use `self.x` or `ClassName.x` |
| `return` vs `print` | `return` = **give the value back**; `print` = **show it on screen only** |
| `or` logic | True if **at least one** side is True |
| `and` logic | True only if **both** sides are True |
| `__init__` | Runs **once**, when the object is born |
| Method | A function that lives **inside** a class and takes `self` |

---

## 📋 Summary Table

| Task | Alignment | Must-fix |
|---|---|---|
| Task 1 — `Car` | ⚠️ Output correct, but `speed` param ignored | `self.speed = speed` |
| Task 2 — `Transaction` | ⚠️ Logic correct, but `print` instead of `return`; only 1 of 3 objects | Use `return`; instantiate 3 objects |

### Corrected Task 2 in full

```python
class Transaction:
    def __init__(self, amount, location, is_fraud):
        self.amount = amount
        self.location = location
        self.is_fraud = is_fraud

    def evaluate_risk(self):
        if self.amount > 10000 or self.location == "International":
            return "Blocked"
        return "Approved"


transactions = [
    Transaction(1000,  "Local",         "No"),
    Transaction(50000, "Local",         "No"),
    Transaction(200,   "International", "No"),
]

for t in transactions:
    print(t.evaluate_risk())
# Approved
# Blocked
# Blocked
```

---

## 🎯 Suggested Follow-up Practice

If you want to cement OOP, try these:

1. **Add `__repr__`** to `Transaction` so printing a transaction shows something useful instead of `<object at 0x...>`.
2. **Add a `deposit()` / `withdraw()`** method to a `BankAccount` class with a guard against negative balance — practice validating `self.amount`.
3. **Turn `evaluate_risk` into a scoring system** — instead of a binary Blocked/Approved, return a risk score (0–100) based on multiple weighted rules.
4. **Subclass it** — create `InternationalTransaction(Transaction)` that overrides `evaluate_risk()` to always Block.
